# 🌍 Ekegusii Multilingual NMT: Google Colab Live Translation Demo
This notebook allows you to test **live machine translation** into **Ekegusii** using fine-tuned **Qwen2.5-7B QLoRA** models loaded directly from **Hugging Face Hub** (`aykgeh/Ekegusii-LLM-Translation`).

### 🏆 Featured Winner Model:
- **`qwen/E10_Model_B_English_Ekegusii`** — 3-Stage Sequential Transfer Model (English → Ekegusii via Kiswahili Bantu Pivot).
- Performance: **16.3 BLEU / 42.9 chrF++**.

## 🔑 Step 1: Hugging Face Authentication (Auto-reads Colab Secrets)
This cell automatically checks for `HF_TOKEN` stored in **Google Colab Secrets** (🔑 icon on the left sidebar).  
⚠️ **Important**: Make sure the toggle switch under **Notebook access** is switched **ON (Blue)** in the Secrets panel!

In [ ]:
from huggingface_hub import login
import os

#@title 🔑 Read Token from Colab Secrets or Manual Entry
HF_TOKEN_INPUT = "" #@param {type:"string"}

hf_token = None
# 1. Try Colab Userdata Secrets
try:
    from google.colab import userdata
    try:
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        try:
            hf_token = userdata.get('HF_TOKE') # Fallback check
        except Exception:
            hf_token = None
except Exception:
    hf_token = None

# 2. Fallback to manual form entry
if not hf_token and HF_TOKEN_INPUT.strip():
    hf_token = HF_TOKEN_INPUT.strip()

if hf_token:
    login(token=hf_token.strip())
    os.environ["HF_TOKEN"] = hf_token.strip()
    print("✅ Hugging Face Authentication Successful!")
else:
    print("ℹ️ No HF_TOKEN detected.")
    print("👉 If your repo is Private, click the key icon (🔑) on the left sidebar in Colab and toggle Notebook Access to ON for HF_TOKEN.")

## 📦 Step 2: Install Dependencies & Setup Environment

In [ ]:
# 1. Install required packages on Colab GPU instance
!pip install -q transformers peft bitsandbytes accelerate datasets streamlit

import os, sys, torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
else:
    print('Warning: No GPU detected! Please enable GPU in Colab (Runtime -> Change runtime type -> T4/A100 GPU).')

## 📂 Step 3: Clone GitHub Repository (Optional)

In [ ]:
# Clone repository if running in fresh Colab session
if not os.path.exists('src'):
    !git clone https://github.com/aykahsay/Ekegusii-LLM-Translation.git repo
    %cd repo
else:
    print('Directory already set up.')

## 🤗 Step 4: Load Model & Tokenizer from Hugging Face

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch, os

# Model identifiers
BASE_MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
HF_REPO_ID = 'aykgeh/Ekegusii-LLM-Translation'
SUBFOLDER = 'qwen/E10_Model_B_English_Ekegusii'  # E10 Model B Winner
token_value = os.environ.get('HF_TOKEN', None) or None

# Explicit GPU device map to prevent CPU offload error on Colab
device_map = {"": 0} if torch.cuda.is_available() else "auto"

print('⏳ 1/3 Loading 4-bit Quantization Config...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True
)

print('⏳ 2/3 Loading Qwen2.5-7B Base Model & Tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, padding_side='left', token=token_value)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map=device_map,
    torch_dtype=torch.bfloat16,
    token=token_value
)

print(f"⏳ 3/3 Attaching E10 Model B Adapter from Hugging Face ('{SUBFOLDER}')...")
model = PeftModel.from_pretrained(base_model, HF_REPO_ID, subfolder=SUBFOLDER, token=token_value)
model.eval()
print('✅ Model Successfully Loaded on Colab GPU!')

## 🚀 Step 5: Live Translation Helper & Benchmark Evaluation

In [ ]:
def translate_sentence(text: str, source_lang: str = 'English', target_lang: str = 'Ekegusii') -> str:
    """Translate a single sentence using the loaded E10 model."""
    prompt = f'<|im_start|>user\nTranslate {source_lang} to {target_lang}:\n{text}<|im_end|>\n<|im_start|>assistant\n'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

# --- Test Public Service Announcements (PSAs) ---
test_psas = [
    'Please wash your hands regularly with clean running water and soap to prevent cholera infection.',
    'Farmers in drought-affected areas are advised to store harvested grain in airtight bags.',
    'Ensure all pregnant women attend early prenatal clinic visits at the nearest county health center.',
    'Residents living in flood-prone riverbanks must evacuate to higher ground immediately.',
    'Stay indoors during severe thunderstorm alerts and avoid standing under tall trees.'
]

print('=' * 70)
print('🌍 EKEGUSII LIVE TRANSLATION RESULTS (E10 Model B)')
print('=' * 70)

for i, sentence in enumerate(test_psas, 1):
    ekegusii_out = translate_sentence(sentence, source_lang='English', target_lang='Ekegusii')
    print(f'\n[{i}] EN:  {sentence}')
    print(f'    EKE: {ekegusii_out}')

print('=' * 70)

## 💬 Step 6: Interactive Form Widget (Colab UI)

In [ ]:
#@title 🌐 Translate Custom Sentence Live { run: 'auto' }
Source_Language = 'English' #@param ['English', 'Kiswahili']
Target_Language = 'Ekegusii' #@param ['Ekegusii', 'Kiswahili', 'English']
Custom_Input = 'Parents are urged to register all school-age children for the upcoming academic term.' #@param {type:'string'}

if Custom_Input.strip():
    result = translate_sentence(Custom_Input, source_lang=Source_Language, target_lang=Target_Language)
    print('=' * 70)
    print(f'INPUT ({Source_Language}): {Custom_Input}')
    print(f'TRANSLATION ({Target_Language}): {result}')
    print('=' * 70)

## 🎛️ Step 7: Test Other Model Adapters (E9, E5, E1, E10 Model C)

In [ ]:
def switch_and_test_adapter(adapter_subfolder: str, test_text: str):
    """Load any model adapter from Hugging Face subfolder and translate."""
    print(f'\n🔄 Switching adapter to "{adapter_subfolder}"...')
    peft_model = PeftModel.from_pretrained(base_model, HF_REPO_ID, subfolder=f'qwen/{adapter_subfolder}', token=token_value)
    peft_model.eval()
    
    prompt = f'<|im_start|>user\nTranslate English to Ekegusii:\n{test_text}<|im_end|>\n<|im_start|>assistant\n'
    inputs = tokenizer(prompt, return_tensors='pt').to(peft_model.device)
    with torch.no_grad():
        out = peft_model.generate(**inputs, max_new_tokens=128, temperature=0.1, do_sample=False)
    res = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    print(f'[{adapter_subfolder}]: {res}')

sample_input = 'Please wash your hands regularly with clean running water and soap.'
print(f'Source Text: {sample_input}\n')

for sub in ['E10_Model_B_English_Ekegusii', 'E10_Model_C_Swahili_Ekegusii', 'E9_Sequential_Transfer', 'E5_Full_Resources', 'E1_English_Ekegusii']:
    switch_and_test_adapter(sub, sample_input)

## 🌐 Step 8: Launch Full Streamlit Web App in Colab

In [ ]:
# Launch Streamlit app in background and expose via localtunnel
!npm install -g localtunnel
!streamlit run app.py & npx localtunnel --port 8501 &

import time
time.sleep(3)
print('🌐 Streamlit app is launching in Google Colab!')
print('1. Click the localtunnel URL generated above.')
print('2. Enter your Colab External IP address to unlock the tunnel:')
!curl ipv4.icanhazip.com